# GABRIEL Document Analysis Tutorial

This notebook demonstrates how to use GABRIEL to analyze text documents.

## Prerequisites

Set environment variables for Azure OpenAI:
```bash
export AZURE_OPENAI_API_KEY="your-api-key"
export AZURE_OPENAI_ENDPOINT="https://your-endpoint.openai.azure.com/"
```

## Step 1: Setup and Imports

In [ ]:
import os
import sys
import pandas as pd

# Add src to path if running from examples directory
sys.path.insert(0, os.path.abspath('../src'))

import gabriel

os.environ["AZURE_OPENAI_API_KEY"] = "THE_KEY"
os.environ["AZURE_OPENAI_ENDPOINT"] = "THE_ENDPOINT"

## Step 2: Load the Markdown Document

In [2]:
# Load the markdown file
doc_path = "2025-11-21 MR AGO.md"

with open(doc_path, "r", encoding="utf-8") as f:
    content = f.read()

print(f"Document length: {len(content)} characters")
print("\n--- Preview (first 1500 chars) ---\n")
print(content[:1500])

Document length: 13325 characters

--- Preview (first 1500 chars) ---

 **To:** 	Dr. Sheetal Singh    
**From:** Team AGORA (AGO)  \- Zavian, Balraj, Luke, Dylan, Alessandro, Kaleb  
**Subject:** Week 12 Management Report – Week Ending December 11, 2025   
**Copies:** Instructor Team

**Key indicators**

|  | 9/19 | 9/26 | 10/03 | 10/10 | 10/17 | Total |
| :---- | :---: | :---: | :---: | :---: | :---: | :---: |
| Precision Execution Score | 10/10 100% | 10/10 100% | 4/7 57.1 | 2/2 100% | 4/4 100% | 30/33 90.9% |

| *Tasks* |  |  |  |  |
| ----- | :---: | ----- | ----- | :---: |
| **BMGT461** | Owner | Est/Act/Assigned | Milestone | Submitted |
| Find Website Host | All | 9/25 | 10/3 | N/A |
| Review the customer discovery module | ZavianDylanLuke | 9/25 | 10/3 | N/A |
| Conduct an Interview with a small start-up (TerraThredz) | Luke | 9/25 | 10/3 | N/A |
| Experiment and learning  | Alessandro | 10/2 | 10/10 | N/A |
| Focus on getting users for site  | All | 10/17 | 10/25 | N/A |
| The

## Step 3: Prepare DataFrame

GABRIEL works with pandas DataFrames. Each row represents one unit of analysis.

In [3]:
# Create DataFrame with document content
df = pd.DataFrame({
    "doc_id": ["MR_AGO_Week12"],
    "content": [content]
})

# Create output directory
output_dir = "mr_analysis_output"
os.makedirs(output_dir, exist_ok=True)

df

,doc_id,content
0,MR_AGO_Week12,**To:** \tDr. Sheetal Singh \n**From:** Te...


## Example A: Extract Structured Information

`gabriel.extract()` pulls out specific facts from text as string/numeric values.

In [ ]:
# Extract key information from the management report
extracted = await gabriel.extract(
    df=df,
    column_name="content",
    attributes={
        "team_name": "What is the team name?",
        "team_members": "List all team member names, separated by commas",
        "report_week": "What week is this report for?",
    },
    save_dir=f"{output_dir}/extraction",
    model="gpt-5.2",
)

# Display results vertically for readability
for col in extracted.columns:
    if col != "content":
        print(f"{col}: {extracted[col].iloc[0]}")

[Extract] Rendering 1 prompts…


2026-01-14 22:35:07,667 - gabriel.utils.openai_utils - WARNING - ⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/


Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~6,138 | Words per prompt: ~6,138
Model: gpt-5.2 | Mode: streaming
Pricing for model 'gpt-5.2': input $1.75/1M, output $14.0/1M
Estimated token usage: input 9,207, output 500 | ~10,207 tokens per call
Estimated synchronous cost: $0.02 (input: $0.02, output: $0.01)

===== Run limits =====
Requests per minute: unknown (API did not share a request limit)
Tokens per minute: unknown (API did not share a token limit)
Approx. words per minute: unknown
We can run up to 650 requests at the same time with your current settings.
⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/

===== Example prompt =====
  Entity:  **To:** 	Dr. Sheetal Singh    
  **From:** Team AGORA (AGO)  \- Zavian, Balraj, Luke, Dylan, Alessandro, Kaleb  
  **Subject:** W

Processing prompts:   0%|          | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-01-14 22:35:07 | Initial parallelization settings: cost_so_far=$0.00, cap=650, active=0, inflight=0, queue=1, processed=0/1, rate_limit_errors=0, throughput<=650 prompts/min


Processing prompts: 100%|██████████| 1/1 [00:03<00:00,  3.89s/it]

[dynamic timeout] Initialized timeout to 9.7s (p90=3.9s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 12,936 tokens, output (incl. reasoning) ≈ 116 tokens.
[token estimate] Updated estimated total cost: ~$0.02 (input $0.02, output $0.00). Updated parallel threads: 508 based on refreshed token usage.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usage tier raises rate limits and allows much faster runs.
Actual total cost: $0.02; average per row: $0.02; average per 1000 rows: $24.26

=== Extraction coverage ===
team_name                                              :     1 extracted,     0 unknown
team_members                                           :     1 extracted,     0 unknown
report_week                                            :     1 extracted,     0 unknown
precision_score                                        :     1 extracted,     0 unknown
num_completed_tasks                       

## Example B: Rate Document Quality

`gabriel.rate()` scores text on attributes using a 0-100 scale.

In [ ]:
# Rate the quality of the management report
ratings = await gabriel.rate(
    df=df,
    column_name="content",
    attributes={
        "clarity": "How clear and easy to understand is the report?",
        "completeness": "How complete is the information? Does it cover all necessary aspects?",
        "organization": "How well organized is the report structure?",
        "professionalism": "How professional is the writing and presentation?",
        "actionability": "How actionable are the items and next steps?",
    },
    save_dir=f"{output_dir}/ratings",
    model="gpt-5.2",
    # use_dummy=True,
)

# Display ratings
rating_cols = ["clarity", "completeness", "organization", "professionalism", "actionability"]
print("Quality Ratings (0-100):")
for col in rating_cols:
    print(f"  {col}: {ratings[col].iloc[0]}")

[Rate] Rendering 1 prompts…


2026-01-14 22:35:17,646 - gabriel.utils.openai_utils - WARNING - ⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/


Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~2,202 | Words per prompt: ~2,202
Model: gpt-5.2 | Mode: streaming
Pricing for model 'gpt-5.2': input $1.75/1M, output $14.0/1M
Estimated token usage: input 3,303, output 500 | ~4,303 tokens per call
Estimated synchronous cost: $0.01 (input: $0.01, output: $0.01)

===== Run limits =====
Requests per minute: unknown (API did not share a request limit)
Tokens per minute: unknown (API did not share a token limit)
Approx. words per minute: unknown
We can run up to 650 requests at the same time with your current settings.
⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/

===== Example prompt =====
  BEGIN TEXT ENTRY
   **To:** 	Dr. Sheetal Singh    
  **From:** Team AGORA (AGO)  \- Zavian, Balraj, Luke, Dylan, Alessandro, Kaleb  
  **Su

Processing prompts:   0%|          | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-01-14 22:35:17 | Initial parallelization settings: cost_so_far=$0.00, cap=650, active=0, inflight=0, queue=1, processed=0/1, rate_limit_errors=0, throughput<=650 prompts/min


Processing prompts: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]

[dynamic timeout] Initialized timeout to 5.9s (p90=2.3s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 4,556 tokens, output (incl. reasoning) ≈ 46 tokens.
[token estimate] Updated estimated total cost: ~$0.01 (input $0.01, output $0.00). Updated parallel threads: 608 based on refreshed token usage.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usage tier raises rate limits and allows much faster runs.
Actual total cost: $0.01; average per row: $0.01; average per 1000 rows: $8.62
Quality Ratings (0-100):
  clarity: 47.0
  completeness: 71.0
  organization: 62.0
  professionalism: 55.0
  actionability: 58.0


## Example C: Analyze Problems Section

Let's extract and classify the problems mentioned in the report.

In [8]:
# Extract just the problems section for detailed analysis
problems_start = content.find("**Problems**")
problems_end = content.find("**Priorities**")
problems_section = content[problems_start:problems_end] if problems_start != -1 else ""

problems_df = pd.DataFrame({
    "section": ["problems"],
    "content": [problems_section]
})

# Rate severity of the problems
problem_analysis = await gabriel.rate(
    df=problems_df,
    column_name="content",
    attributes={
        "severity": "How severe are the problems described? (100 = critical blockers, 0 = minor issues)",
        "solvability": "How solvable are these problems with available resources? (100 = easily solvable, 0 = very difficult)",
        "impact_on_timeline": "How much might these problems impact the project timeline? (100 = major delays, 0 = no impact)",
    },
    save_dir=f"{output_dir}/problem_analysis",
    model="gpt-5.2",
    # use_dummy=True,
)

print("Problem Analysis (0-100):")
for col in ["severity", "solvability", "impact_on_timeline"]:
    print(f"  {col}: {problem_analysis[col].iloc[0]}")

[Rate] Rendering 1 prompts…


2026-01-14 22:35:35,980 - gabriel.utils.openai_utils - WARNING - ⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/


Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~539 | Words per prompt: ~539
Model: gpt-5.2 | Mode: streaming
Pricing for model 'gpt-5.2': input $1.75/1M, output $14.0/1M
Estimated token usage: input 808, output 500 | ~1,808 tokens per call
Estimated synchronous cost: $0.01 (input: $0.00, output: $0.01)

===== Run limits =====
Requests per minute: unknown (API did not share a request limit)
Tokens per minute: unknown (API did not share a token limit)
Approx. words per minute: unknown
We can run up to 650 requests at the same time with your current settings.
⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/

===== Example prompt =====
  BEGIN TEXT ENTRY
  **Problems**

  We are aware of these problems:

  | Owner   (Due Date) | What |
  | ----- | :---- |
  | All | Our initial fi

Processing prompts:   0%|          | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-01-14 22:35:35 | Initial parallelization settings: cost_so_far=$0.00, cap=650, active=0, inflight=0, queue=1, processed=0/1, rate_limit_errors=0, throughput<=650 prompts/min


Processing prompts: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]

[dynamic timeout] Initialized timeout to 4.2s (p90=1.7s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 768 tokens, output (incl. reasoning) ≈ 32 tokens.
[token estimate] Updated estimated total cost: ~$0.00 (input $0.00, output $0.00). Updated parallel threads: 650 based on refreshed token usage.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usage tier raises rate limits and allows much faster runs.
Actual total cost: $0.00; average per row: $0.00; average per 1000 rows: $1.79
Problem Analysis (0-100):
  severity: 71.0
  solvability: 62.0
  impact_on_timeline: 68.0


## Example D: Custom Analysis with `whatever()`

`gabriel.whatever()` allows fully custom prompts for flexible analysis.

In [17]:
# Custom executive summary analysis
custom = await gabriel.whatever(
      df=df,
      column_name="content",
      prompts="""
      Analyze this management report and respond in JSON format with:
      - executive_summary: 2-3 sentence summary of the report
      - key_achievements: List of top 5 accomplishments
      - critical_risks: List of top 3 risks or problems
      - recommendations: List of 3 actionable recommendations for the team
      - overall_health_score: Project health score from 1-10
      
      Return valid JSON only.
      """,
      save_dir=f"{output_dir}/custom",
      model="gpt-5.2",
      json_mode=False,
)

# Pretty print the JSON response
import json
response = custom["Response"].iloc[0]
try:
    parsed = json.loads(response)
    print(json.dumps(parsed, indent=2))
except:
    print(response)

2026-01-14 22:47:53,024 - gabriel.utils.openai_utils - WARNING - ⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/


Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~1,856 | Words per prompt: ~1,856
Model: gpt-5.2 | Mode: streaming
Pricing for model 'gpt-5.2': input $1.75/1M, output $14.0/1M
Estimated token usage: input 2,784, output 500 | ~3,784 tokens per call
Estimated synchronous cost: $0.01 (input: $0.00, output: $0.01)
Reading from existing files at mr_analysis_output/custom/custom_prompt_responses.csv...
Loaded 1 rows; 0 already marked complete.

===== Run limits =====
Requests per minute: unknown (API did not share a request limit)
Tokens per minute: unknown (API did not share a token limit)
Approx. words per minute: unknown
We can run up to 650 requests at the same time with your current settings.
⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/

===== Example prompt =====
   **To:** 

Processing prompts:   0%|          | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-01-14 22:47:53 | Initial parallelization settings: cost_so_far=$0.00, cap=650, active=0, inflight=0, queue=1, processed=0/1, rate_limit_errors=0, throughput<=650 prompts/min


Processing prompts: 100%|██████████| 1/1 [00:44<00:00, 44.15s/it]

[dynamic timeout] Initialized timeout to 110.4s (p90=44.1s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 4,020 tokens, output (incl. reasoning) ≈ 1,553 tokens.
[token estimate] Updated estimated total cost: ~$0.03 (input $0.01, output $0.02). Updated parallel threads: 441 based on refreshed token usage.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usage tier raises rate limits and allows much faster runs.
Actual total cost: $0.00; average per row: $0.00; average per 1000 rows: $4.87



## Example E: Analyze Team Member Contributions

Split progress items by team member and analyze their contributions.

In [36]:
# Create a DataFrame with individual team members
team_members = ["Zavian", "Balraj", "Luke", "Dylan", "Alessandro", "Kaleb"]

member_df = pd.DataFrame({
    "member": team_members,
    "report_content": [content] * len(team_members)
})

# Extract each member's contributions
contributions = await gabriel.extract(
    df=member_df,
    column_name="report_content",
    attributes={
        "tasks_completed": f"Based on the 'member' column value, list all tasks this specific team member completed",
        "tasks_assigned": f"Based on the 'member' column value, list all tasks assigned to this specific team member",
    },
    additional_instructions="Focus only on tasks associated with the team member name in the 'member' column",
    save_dir=f"{output_dir}/contributions",
    model="gpt-5.2",
    # use_dummy=True,
)

contributions[["member", "tasks_completed", "tasks_assigned"]]

[Extract] Rendering 1 prompts…


2026-01-14 23:23:53,735 - gabriel.utils.openai_utils - WARNING - ⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/


Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~6,145 | Words per prompt: ~6,145
Model: gpt-5.2 | Mode: streaming
Pricing for model 'gpt-5.2': input $1.75/1M, output $14.0/1M
Estimated token usage: input 9,217, output 500 | ~10,217 tokens per call
Estimated synchronous cost: $0.02 (input: $0.02, output: $0.01)

===== Run limits =====
Requests per minute: unknown (API did not share a request limit)
Tokens per minute: unknown (API did not share a token limit)
Approx. words per minute: unknown
We can run up to 650 requests at the same time with your current settings.
⚠️ API did not return complete rate-limit headers. Running with conservative defaults. If you are on a free/low-balance plan, add funds to avoid quota blocks: https://platform.openai.com/settings/organization/billing/

===== Example prompt =====
  Entity:  **To:** 	Dr. Sheetal Singh    
  **From:** Team AGORA (AGO)  \- Zavian, Balraj, Luke, Dylan, Alessandro, Kaleb  
  **Subject:** W

Processing prompts:   0%|          | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-01-14 23:23:53 | Initial parallelization settings: cost_so_far=$0.00, cap=650, active=0, inflight=0, queue=1, processed=0/1, rate_limit_errors=0, throughput<=650 prompts/min


Processing prompts: 100%|██████████| 1/1 [00:06<00:00,  6.74s/it]

[dynamic timeout] Initialized timeout to 16.8s (p90=6.7s, factor=2.50).
[token estimate] Refreshed per-prompt estimates from observed usage (1 sample): input ≈ 12,918 tokens, output (incl. reasoning) ≈ 51 tokens.
[token estimate] Updated estimated total cost: ~$0.02 (input $0.02, output $0.00). Updated parallel threads: 512 based on refreshed token usage.
[token estimate] Updated time estimate: minimum of 1 minute. Moving to a higher usage tier raises rate limits and allows much faster runs.
Actual total cost: $0.02; average per row: $0.02; average per 1000 rows: $23.32

=== Extraction coverage ===
tasks_completed                                        :     0 extracted,     1 unknown
tasks_assigned                                         :     0 extracted,     1 unknown



,member,tasks_completed,tasks_assigned
0,Zavian,<NA>,<NA>
1,Balraj,<NA>,<NA>
2,Luke,<NA>,<NA>
3,Dylan,<NA>,<NA>
4,Alessandro,<NA>,<NA>
5,Kaleb,<NA>,<NA>


## Tips and Best Practices

### Testing Mode
Use `use_dummy=True` to test your pipeline without consuming API credits:
```python
result = await gabriel.rate(..., use_dummy=True)
```

### Checkpointing
GABRIEL automatically saves intermediate results to `save_dir`. If interrupted, re-running resumes from checkpoint. Use `reset_files=True` to start fresh:
```python
result = await gabriel.rate(..., reset_files=True)
```

### Multiple Runs for Reliability
Use `n_runs` to get averaged scores across multiple API calls:
```python
result = await gabriel.rate(..., n_runs=3)
```

## GABRIEL Function Reference

| Function | Purpose | Output |
|----------|---------|--------|
| `extract()` | Pull structured facts from text | Strings/numbers |
| `rate()` | Score text on attributes | 0-100 scores |
| `classify()` | Assign text to categories | Label names |
| `whatever()` | Custom prompt analysis | Any format |
| `rank()` | Rank items by criteria | Rankings |
| `deduplicate()` | Find duplicate entries | Cluster IDs |
| `filter()` | Filter rows by criteria | Boolean mask |

### Modality Support

GABRIEL supports multiple input types via the `modality` parameter:

| Modality | Input Format |
|----------|-------------|
| `"text"` | Plain text (default) |
| `"image"` | Image file paths or URLs |
| `"audio"` | Audio file paths |
| `"web"` | Web URLs (auto-fetches content) |
| `"entity"` | Entity names for extraction |